In [1]:
# Install first:
# pip install dukascopy-python pandas numpy tqdm

from pathlib import Path
from datetime import datetime
import time

import numpy as np
import pandas as pd
from tqdm import tqdm

import dukascopy_python
from dukascopy_python.instruments import INSTRUMENT_FX_MAJORS_USD_JPY


# -----------------------------
# Settings
# -----------------------------

START_DATE = datetime(1998, 12, 14)
END_DATE = datetime(2026, 7, 2)
# 2 July is used as the end boundary so that 1 July is included.

OUTPUT_DIR = Path("usdjpy_15min_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BID_FILE = OUTPUT_DIR / "usdjpy_15min_bid.csv"
ASK_FILE = OUTPUT_DIR / "usdjpy_15min_ask.csv"
MID_FILE = OUTPUT_DIR / "usdjpy_15min_mid.csv"


# -----------------------------
# Download function
# -----------------------------

def download_data(start, end, offer_side):
    """
    Download 15-minute USD/JPY OHLC data for a date range.
    """

    return dukascopy_python.fetch(
        INSTRUMENT_FX_MAJORS_USD_JPY,
        dukascopy_python.INTERVAL_MIN_15,
        offer_side,
        start,
        end,
    )


# -----------------------------
# Download yearly chunks
# -----------------------------

bid_data = []
ask_data = []

for year in tqdm(range(1998, 2027), desc="Downloading yearly data"):

    chunk_start = max(
        START_DATE,
        datetime(year, 1, 1)
    )

    chunk_end = min(
        END_DATE,
        datetime(year + 1, 1, 1)
    )

    if chunk_start >= chunk_end:
        continue

    print(f"\nDownloading {chunk_start.date()} to {chunk_end.date()}")

    try:
        bid_chunk = download_data(
            chunk_start,
            chunk_end,
            dukascopy_python.OFFER_SIDE_BID
        )

        ask_chunk = download_data(
            chunk_start,
            chunk_end,
            dukascopy_python.OFFER_SIDE_ASK
        )

        if bid_chunk is not None and not bid_chunk.empty:
            bid_data.append(bid_chunk)

        if ask_chunk is not None and not ask_chunk.empty:
            ask_data.append(ask_chunk)

        # Avoid sending requests too quickly.
        time.sleep(1)

    except Exception as error:
        print(f"Failed for {year}: {error}")
        continue


# -----------------------------
# Combine BID data
# -----------------------------

if not bid_data:
    raise RuntimeError("No bid data was downloaded.")

if not ask_data:
    raise RuntimeError("No ask data was downloaded.")

bid = pd.concat(bid_data)
ask = pd.concat(ask_data)

# Ensure that the index is interpreted as UTC timestamps.
bid.index = pd.to_datetime(bid.index, utc=True)
ask.index = pd.to_datetime(ask.index, utc=True)

# Sort chronologically.
bid = bid.sort_index()
ask = ask.sort_index()

# Remove duplicate timestamps.
bid = bid[~bid.index.duplicated(keep="first")]
ask = ask[~ask.index.duplicated(keep="first")]

# Restrict exactly to the requested period.
bid = bid.loc[
    (bid.index >= pd.Timestamp(START_DATE, tz="UTC")) &
    (bid.index < pd.Timestamp(END_DATE, tz="UTC"))
]

ask = ask.loc[
    (ask.index >= pd.Timestamp(START_DATE, tz="UTC")) &
    (ask.index < pd.Timestamp(END_DATE, tz="UTC"))
]

# Save the separate bid and ask data.
bid.to_csv(BID_FILE)
ask.to_csv(ASK_FILE)


# -----------------------------
# Calculate midpoint prices
# -----------------------------

# Keep only timestamps present in both datasets.
common_index = bid.index.intersection(ask.index)

bid = bid.loc[common_index]
ask = ask.loc[common_index]

mid = pd.DataFrame(index=common_index)

# Calculate midpoint OHLC values.
for column in ["open", "high", "low", "close"]:
    mid[f"mid_{column}"] = (
        bid[column] + ask[column]
    ) / 2

# Calculate midpoint spread information where possible.
mid["spread_close"] = ask["close"] - bid["close"]

# Calculate the midpoint log return using the 15-minute close.
mid["log_return"] = np.log(
    mid["mid_close"] / mid["mid_close"].shift(1)
)

# Remove the first observation, which has no return.
mid = mid.dropna(subset=["mid_close", "log_return"])


# -----------------------------
# Save the midpoint data
# -----------------------------

mid.to_csv(MID_FILE)

print("\nFinished.")
print(f"Observations: {len(mid):,}")
print(f"First timestamp: {mid.index.min()}")
print(f"Last timestamp:  {mid.index.max()}")
print(f"Bid data saved to: {BID_FILE}")
print(f"Ask data saved to: {ASK_FILE}")
print(f"Mid data saved to: {MID_FILE}")

INFO:DUKASCRIPT:current timestamp :2007-04-26T09:15:00
INFO:DUKASCRIPT:current timestamp :2007-04-26T09:15:00

INFO:DUKASCRIPT:current timestamp :2012-01-30T21:30:00
INFO:DUKASCRIPT:current timestamp :2012-02-20T19:45:00
INFO:DUKASCRIPT:current timestamp :2012-03-12T16:00:00
INFO:DUKASCRIPT:current timestamp :2012-04-02T20:00:00
INFO:DUKASCRIPT:current timestamp :2012-04-25T08:30:00
INFO:DUKASCRIPT:current timestamp :2012-05-16T04:30:00
INFO:DUKASCRIPT:current timestamp :2012-06-06T02:00:00
INFO:DUKASCRIPT:current timestamp :2012-06-26T21:45:00
INFO:DUKASCRIPT:current timestamp :2012-07-19T21:30:00
INFO:DUKASCRIPT:current timestamp :2012-08-09T18:30:00
INFO:DUKASCRIPT:current timestamp :2012-08-30T17:15:00
INFO:DUKASCRIPT:current timestamp :2012-09-20T13:15:00
INFO:DUKASCRIPT:current timestamp :2012-10-11T10:30:00
INFO:DUKASCRIPT:current timestamp :2012-11-01T07:45:00
INFO:DUKASCRIPT:current timestamp :2012-11-22T03:30:00
INFO:DUKASCRIPT:current timestamp :2012-12-14T05:45:00
INFO:DUKASCRIPT:current timestamp :2012-01-30T21:30:00
INFO:DUKASCRIPT:current timestamp :2012-02-20T19:45:00
INFO:DUKAS

INFO:DUKASCRIPT:current timestamp :2013-01-21T19:45:00
INFO:DUKASCRIPT:current timestamp :2013-02-12T08:45:00
INFO:DUKASCRIPT:current timestamp :2013-03-06T03:00:00
INFO:DUKASCRIPT:current timestamp :2013-03-26T23:00:00
INFO:DUKASCRIPT:current timestamp :2013-04-16T22:15:00
INFO:DUKASCRIPT:current timestamp :2013-05-07T18:45:00
INFO:DUKASCRIPT:current timestamp :2013-05-29T01:00:00
INFO:DUKASCRIPT:current timestamp :2013-06-19T03:30:00
INFO:DUKASCRIPT:current timestamp :2013-07-10T01:00:00
INFO:DUKASCRIPT:current timestamp :2013-07-30T21:45:00
INFO:DUKASCRIPT:current timestamp :2013-08-20T18:45:00
INFO:DUKASCRIPT:current timestamp :2013-09-10T17:15:00
INFO:DUKASCRIPT:current timestamp :2013-10-01T13:30:00
INFO:DUKASCRIPT:current timestamp :2013-10-22T09:30:00
INFO:DUKASCRIPT:current timestamp :2013-11-12T07:00:00
INFO:DUKASCRIPT:current timestamp :2013-12-03T04:15:00
INFO:DUKASCRIPT:current timestamp :2013-12-24T00:15:00
INFO:DUKASCRIPT:current timestamp :2013-01-21T19:45:00
INFO:DUKAS

INFO:DUKASCRIPT:current timestamp :2014-01-21T23:30:00
INFO:DUKASCRIPT:current timestamp :2014-02-12T04:00:00
INFO:DUKASCRIPT:current timestamp :2014-03-05T04:45:00
INFO:DUKASCRIPT:current timestamp :2014-03-26T00:30:00
INFO:DUKASCRIPT:current timestamp :2014-04-15T21:30:00
INFO:DUKASCRIPT:current timestamp :2014-05-06T23:30:00
INFO:DUKASCRIPT:current timestamp :2014-06-04T20:45:00
INFO:DUKASCRIPT:current timestamp :2014-07-03T17:45:00
INFO:DUKASCRIPT:current timestamp :2014-08-01T16:30:00
INFO:DUKASCRIPT:current timestamp :2014-09-01T13:45:00
INFO:DUKASCRIPT:current timestamp :2014-09-30T10:00:00
INFO:DUKASCRIPT:current timestamp :2014-10-29T04:45:00
INFO:DUKASCRIPT:current timestamp :2014-11-27T01:30:00
INFO:DUKASCRIPT:current timestamp :2014-12-26T12:00:00
INFO:DUKASCRIPT:current timestamp :2014-01-21T23:30:00
INFO:DUKASCRIPT:current timestamp :2014-02-12T04:00:00
INFO:DUKASCRIPT:current timestamp :2014-03-05T04:45:00
INFO:DUKASCRIPT:current timestamp :2014-03-26T00:30:00
INFO:DUKAS

INFO:DUKASCRIPT:current timestamp :2015-01-30T17:45:00
INFO:DUKASCRIPT:current timestamp :2015-03-02T13:45:00
INFO:DUKASCRIPT:current timestamp :2015-03-31T09:45:00
INFO:DUKASCRIPT:current timestamp :2015-04-29T06:15:00
INFO:DUKASCRIPT:current timestamp :2015-05-28T03:45:00
INFO:DUKASCRIPT:current timestamp :2015-06-26T00:00:00
INFO:DUKASCRIPT:current timestamp :2015-07-24T21:00:00
INFO:DUKASCRIPT:current timestamp :2015-08-24T18:45:00
INFO:DUKASCRIPT:current timestamp :2015-09-22T15:30:00
INFO:DUKASCRIPT:current timestamp :2015-10-21T12:00:00
INFO:DUKASCRIPT:current timestamp :2015-11-19T08:15:00
INFO:DUKASCRIPT:current timestamp :2015-12-18T04:30:00
INFO:DUKASCRIPT:current timestamp :2015-01-30T17:45:00
INFO:DUKASCRIPT:current timestamp :2015-03-02T13:45:00
INFO:DUKASCRIPT:current timestamp :2015-03-31T09:45:00
INFO:DUKASCRIPT:current timestamp :2015-04-29T06:15:00
INFO:DUKASCRIPT:current timestamp :2015-05-28T03:45:00
INFO:DUKASCRIPT:current timestamp :2015-06-26T00:00:00
INFO:DUKAS

...

INFO:DUKASCRIPT:current timestamp :2023-01-30T21:00:00
INFO:DUKASCRIPT:current timestamp :2023-02-28T18:30:00
INFO:DUKASCRIPT:current timestamp :2023-03-29T15:45:00
INFO:DUKASCRIPT:current timestamp :2023-04-27T13:45:00
INFO:DUKASCRIPT:current timestamp :2023-05-26T10:30:00
INFO:DUKASCRIPT:current timestamp :2023-06-26T08:15:00
INFO:DUKASCRIPT:current timestamp :2023-07-25T06:00:00
INFO:DUKASCRIPT:current timestamp :2023-08-23T02:45:00
INFO:DUKASCRIPT:current timestamp :2023-09-21T00:00:00
INFO:DUKASCRIPT:current timestamp :2023-10-19T20:30:00
INFO:DUKASCRIPT:current timestamp :2023-11-17T17:30:00
INFO:DUKASCRIPT:current timestamp :2023-12-18T14:30:00
INFO:DUKASCRIPT:current timestamp :2023-01-30T21:00:00
INFO:DUKASCRIPT:current timestamp :2023-02-28T18:30:00
INFO:DUKASCRIPT:current timestamp :2023-03-29T15:45:00
INFO:DUKASCRIPT:current timestamp :2023-04-27T13:45:00
INFO:DUKASCRIPT:current timestamp :2023-05-26T10:30:00
INFO:DUKASCRIPT:current timestamp :2023-06-26T08:15:00
INFO:DUKAS

INFO:DUKASCRIPT:current timestamp :2024-01-30T18:30:00
INFO:DUKASCRIPT:current timestamp :2024-02-28T15:30:00
INFO:DUKASCRIPT:current timestamp :2024-03-28T11:45:00
INFO:DUKASCRIPT:current timestamp :2024-04-26T09:45:00
INFO:DUKASCRIPT:current timestamp :2024-05-27T07:30:00
INFO:DUKASCRIPT:current timestamp :2024-06-25T05:15:00
INFO:DUKASCRIPT:current timestamp :2024-07-24T02:45:00
INFO:DUKASCRIPT:current timestamp :2024-08-21T23:30:00
INFO:DUKASCRIPT:current timestamp :2024-09-19T21:15:00
INFO:DUKASCRIPT:current timestamp :2024-10-18T20:00:00
INFO:DUKASCRIPT:current timestamp :2024-11-18T17:45:00
INFO:DUKASCRIPT:current timestamp :2024-12-17T17:45:00
INFO:DUKASCRIPT:current timestamp :2024-01-30T18:30:00
INFO:DUKASCRIPT:current timestamp :2024-02-28T15:30:00
INFO:DUKASCRIPT:current timestamp :2024-03-28T11:45:00
INFO:DUKASCRIPT:current timestamp :2024-04-26T09:45:00
INFO:DUKASCRIPT:current timestamp :2024-05-27T07:30:00
INFO:DUKASCRIPT:current timestamp :2024-06-25T05:15:00
INFO:DUKAS

INFO:DUKASCRIPT:current timestamp :2025-01-30T20:00:00
INFO:DUKASCRIPT:current timestamp :2025-02-28T17:45:00
INFO:DUKASCRIPT:current timestamp :2025-03-31T16:30:00
INFO:DUKASCRIPT:current timestamp :2025-04-29T15:30:00
INFO:DUKASCRIPT:current timestamp :2025-05-28T13:00:00
INFO:DUKASCRIPT:current timestamp :2025-06-26T10:30:00
INFO:DUKASCRIPT:current timestamp :2025-07-25T08:45:00
INFO:DUKASCRIPT:current timestamp :2025-08-25T06:45:00
INFO:DUKASCRIPT:current timestamp :2025-09-23T04:45:00
INFO:DUKASCRIPT:current timestamp :2025-10-22T02:15:00
INFO:DUKASCRIPT:current timestamp :2025-11-19T23:45:00
INFO:DUKASCRIPT:current timestamp :2025-12-18T20:45:00
INFO:DUKASCRIPT:current timestamp :2025-01-30T20:00:00
INFO:DUKASCRIPT:current timestamp :2025-02-28T17:45:00
INFO:DUKASCRIPT:current timestamp :2025-03-31T16:30:00
INFO:DUKASCRIPT:current timestamp :2025-04-29T15:30:00
INFO:DUKASCRIPT:current timestamp :2025-05-28T13:00:00
INFO:DUKASCRIPT:current timestamp :2025-06-26T10:30:00
INFO:DUKAS

INFO:DUKASCRIPT:current timestamp :2026-01-30T19:30:00
INFO:DUKASCRIPT:current timestamp :2026-03-02T16:15:00
INFO:DUKASCRIPT:current timestamp :2026-03-31T13:45:00
INFO:DUKASCRIPT:current timestamp :2026-04-29T12:30:00
INFO:DUKASCRIPT:current timestamp :2026-05-28T10:45:00
INFO:DUKASCRIPT:current timestamp :2026-06-26T09:45:00
INFO:DUKASCRIPT:current timestamp :2026-01-30T19:30:00
INFO:DUKASCRIPT:current timestamp :2026-03-02T16:15:00
INFO:DUKASCRIPT:current timestamp :2026-03-31T13:45:00
INFO:DUKASCRIPT:current timestamp :2026-04-29T12:30:00
INFO:DUKASCRIPT:current timestamp :2026-05-28T10:45:00
INFO:DUKASCRIPT:current timestamp :2026-06-26T09:45:00



Finished.
Observations: 386,114
First timestamp: 2007-03-27 00:15:00+00:00
Last timestamp:  2026-07-01 23:00:00+00:00
Bid data saved to: usdjpy_15min_data\usdjpy_15min_bid.csv
Ask data saved to: usdjpy_15min_data\usdjpy_15min_ask.csv
Mid data saved to: usdjpy_15min_data\usdjpy_15min_mid.csv
